# Add New Data to Train Split

Loads the **existing** dataset from HuggingFace, then ingests new JSONL files and appends them **only to the train split** — the test set is never touched.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
HF_DATASET   = "alexneakameni/ZSHOT-HARDSET-v2"
NEW_FILES_PATH = "../data/wikipedia/New/*/wiki*.jsonl"   # <── adjust as needed
# ──────────────────────────────────────────────────────────────────────────────

In [2]:
import datasets

existing = datasets.load_dataset(HF_DATASET)
existing

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 661922
    })
    test: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 3563
    })
})

In [3]:
import pandas as pd

df_train_existing = existing["train"].to_pandas()
df_test           = existing["test"].to_pandas()

# Reconstruct the held-out test labels:
# These are labels that appear as POSITIVE in the test set
# but were NEVER seen (positive or negative) in train.
# This matches the original strict split definition.
train_all_labels = (
    set(lab for labs in df_train_existing["labels"]     for lab in labs)
    | set(lab for labs in df_train_existing["not_labels"] for lab in labs)
)
test_pos_labels = set(lab for labs in df_test["labels"] for lab in labs)
test_labels = test_pos_labels - train_all_labels

print(f"Existing train rows : {len(df_train_existing)}")
print(f"Existing test rows  : {len(df_test)}")
print(f"Reconstructed held-out test label vocab: {len(test_labels)}")


Existing train rows : 661922
Existing test rows  : 3563
Reconstructed held-out test label vocab: 2829


In [4]:
import json
from glob import glob
from pathlib import Path

files = sorted(glob(NEW_FILES_PATH))
print(f"New files found: {len(files)}")
for f in files:
    print(" ", f)

New files found: 1
  ../data/wikipedia/New/Gemma4E4B/wikipedia_synthetic.jsonl


In [5]:
def load_jsonl(file):
    model = Path(file).parent.name
    rows = []
    with open(file) as fh:
        for line in fh:
            d = json.loads(line)
            rows.append({
                "text":       d["text"],
                "labels":     d.get("labels", []),
                "not_labels": d.get("not_labels", []),
                "model":      model,
            })
    return rows

raw = [row for f in files for row in load_jsonl(f)]
df_new = pd.DataFrame(raw)
print(f"Raw new rows loaded: {len(df_new)}")

Raw new rows loaded: 247555


In [6]:
# Merge duplicate texts (same logic as CreateDataset)
def merge_group(group):
    merged_labels     = set().union(*group["labels"])
    merged_not_labels = set().union(*group["not_labels"])
    merged_not_labels -= merged_labels
    models = sorted(set(group["model"]))
    return pd.Series({
        "labels":     sorted(merged_labels),
        "not_labels": sorted(merged_not_labels),
        "model":      models if len(models) > 1 else models[0],
    })

df_new = (
    df_new.groupby("text", sort=False)
    .apply(merge_group, include_groups=False)
    .reset_index()
)
print(f"Unique new texts after merging: {len(df_new)}")

Unique new texts after merging: 247532


In [7]:
# Mirror the original strict split logic:
#   test  → rows where labels intersects test_labels (positive hit)
#   train → rows where neither labels nor not_labels intersects test_labels
#   drop  → rows where only not_labels intersects test_labels (contaminated)

pos_hit  = df_new["labels"].apply(lambda labs: bool(set(labs) & test_labels))
neg_hit  = df_new["not_labels"].apply(lambda labs: bool(set(labs) & test_labels))

df_new_for_test  = df_new[pos_hit].copy()
df_new_for_train = df_new[~pos_hit & ~neg_hit].copy()
dropped          = df_new[~pos_hit & neg_hit]

print(f"New rows → test    : {len(df_new_for_test)}")
print(f"New rows → train   : {len(df_new_for_train)}")
print(f"New rows → dropped : {len(dropped)}  (test label in not_labels only)")

# Deduplicate against existing train
existing_train_texts = set(df_train_existing["text"])
novel_train_mask     = ~df_new_for_train["text"].isin(existing_train_texts)
df_new_novel_train   = df_new_for_train[novel_train_mask].copy()
print(f"\nNew train rows (deduped): {len(df_new_novel_train)} "
      f"(skipped {(~novel_train_mask).sum()} already in train)")

# Deduplicate against existing test
existing_test_texts = set(df_test["text"])
novel_test_mask     = ~df_new_for_test["text"].isin(existing_test_texts)
df_new_novel_test   = df_new_for_test[novel_test_mask].copy()
print(f"New test  rows (deduped): {len(df_new_novel_test)} "
      f"(skipped {(~novel_test_mask).sum()} already in test)")


New rows → test    : 1302
New rows → train   : 242802
New rows → dropped : 3428  (test label in not_labels only)

New train rows (deduped): 242770 (skipped 32 already in train)
New test  rows (deduped): 1302 (skipped 0 already in test)


In [8]:
# Append new rows to the appropriate split
df_train_updated = pd.concat([df_train_existing, df_new_novel_train], ignore_index=True)
df_test_updated  = pd.concat([df_test,           df_new_novel_test],  ignore_index=True)

print(f"Updated train size: {len(df_train_existing)} → {len(df_train_updated)} rows")
print(f"Updated test  size: {len(df_test)}           → {len(df_test_updated)}  rows")


Updated train size: 661922 → 904692 rows
Updated test  size: 3563           → 4865  rows


In [9]:
train_ds = datasets.Dataset.from_pandas(df_train_updated)
test_ds  = datasets.Dataset.from_pandas(df_test_updated)

dataset = datasets.DatasetDict({
    "train": train_ds,
    "test":  test_ds,
})
dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 904692
    })
    test: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 4865
    })
})

In [10]:
import pandas as pd
from IPython.display import display

train_pos = set(lab for labs in df_train_updated["labels"]     for lab in labs)
train_neg = set(lab for labs in df_train_updated["not_labels"] for lab in labs)
test_pos  = set(lab for labs in df_test_updated["labels"]      for lab in labs)
test_neg  = set(lab for labs in df_test_updated["not_labels"]  for lab in labs)

total = len(test_pos) + len(test_neg)

# For a given target set, split by how the label was seen in train
def bucket(s):
    return {
        "seen in train as positive only":          len((s & train_pos) - train_neg),
        "seen in train as negative only":          len((s & train_neg) - train_pos),
        "seen in train as both pos and neg":       len(s & train_pos & train_neg),
        "never seen in train":                     len(s - train_pos - train_neg),
    }

tp = bucket(test_pos)   # labels used as ground-truth positives in test
tn = bucket(test_neg)   # labels used as hard negatives in test
index = list(tp.keys())

counts = pd.DataFrame({
    "used as positive in test": [tp[k] for k in index],
    "used as negative in test": [tn[k] for k in index],
    "total":                    [tp[k] + tn[k] for k in index],
}, index=index)
counts.index.name = "how the label was seen in train"

ratios = (counts / total * 100).round(1).astype(str) + "%"

print(f"Unique test positive labels: {len(test_pos)}   "
      f"Unique test negative labels: {len(test_neg)}   "
      f"Total: {total}\n")
print("── Counts ──")
display(counts)
print("── % of total test labels ──")
display(ratios)


Unique test positive labels: 12166   Unique test negative labels: 24826   Total: 36992

── Counts ──


,used as positive in test,used as negative in test,total
how the label was seen in train,,,
seen in train as positive only,404,870,1274
seen in train as negative only,844,1201,2045
seen in train as both pos and neg,7347,16485,23832
never seen in train,3571,6270,9841


── % of total test labels ──


,used as positive in test,used as negative in test,total
how the label was seen in train,,,
seen in train as positive only,1.1%,2.4%,3.4%
seen in train as negative only,2.3%,3.2%,5.5%
seen in train as both pos and neg,19.9%,44.6%,64.4%
never seen in train,9.7%,16.9%,26.6%


In [11]:
dataset.push_to_hub(
    HF_DATASET,
    commit_description=(
        f"Add {len(df_new_novel_train)} new train rows, "
        f"{len(df_new_novel_test)} new test rows."
    ),
)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2/commit/74180b49660c2b0ee04b5d78a4323d70ee92423b', commit_message='Upload dataset', commit_description='Add 242770 new train rows, 1302 new test rows.', oid='74180b49660c2b0ee04b5d78a4323d70ee92423b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alexneakameni/ZSHOT-HARDSET-v2'), pr_revision=None, pr_num=None)